# Load-out-of-store vs direct-path timing

How much does going *through the store* cost versus just reading a file that is already sitting at a known path? The whole store &mdash; metadata, lineage and artifact bytes &mdash; is a single `ancestree.db` file with a content-addressed chunk pool inside it, so a read has to find the node, reassemble the artifact from its chunks and hand back a path. This notebook times **every step** of that:

1. **finding** the node (by id, and by metadata search),
2. **discovering** its artifact with `.artifacts()` (which, on a cold cache, also *reassembles* the file from the chunk pool),
3. **reading / parsing** the returned path,

&mdash; and compares the total against `pd.read_csv(path)` on a plain copy of the same file. Because reads route through a session cache, the store path has two regimes: **cold** (first touch &mdash; reassemble from the pool) and **warm** (cache hit &mdash; a plain file read). Both are measured.

Everything lives in a temp folder, so re-running top to bottom starts clean and leaves no litter behind.

In [1]:
import shutil
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd

import ancestree

# A temp workdir keeps the executed notebook from leaving a store behind;
# point ROOT at any real directory in your own work.
WORKDIR = Path(tempfile.mkdtemp(prefix="ancestree-loadbench-"))
ROOT = WORKDIR / "load_benchmark_store"  # the lineage store (chunk=True)
BASE = WORKDIR / "load_benchmark_direct"  # plain copies of the same files, no store
BASE.mkdir(parents=True)

store = ancestree.LineageStore(ROOT, dedup=True, chunk=True)


def timed(label, fn, reps=50):
    ts = []
    for _ in range(reps):
        t = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t) * 1000)
    print(f"{label:<44} median {np.median(ts):8.3f} ms   (best {min(ts):.3f})")
    return float(np.median(ts))

## 1. Build a store with real artifacts (+ plain copies for the baseline)

Most nodes are metadata-only (to give the index a realistic size for the search step); a handful hold a ~2 MB CSV. Each artifact is also written verbatim to `BASE/` so we can read the *identical bytes* directly, with no store involved.

In [2]:
N_TOTAL = 1000  # nodes in the store (search scans the whole index)
N_ART = 30  # of those, how many hold a real artifact we will load
ROWS = 95_000  # ~2 MB per CSV
rng = np.random.default_rng(0)


def make_csv(nrows):
    return pd.DataFrame(
        {
            "ts": np.arange(nrows),
            "a": rng.normal(size=nrows).round(4),
            "b": rng.normal(size=nrows).round(4),
            "label": rng.integers(0, 3, nrows),
        }
    )


target_ids, direct_paths = [], []
t0 = time.perf_counter()
for i in range(N_TOTAL):
    with store.create_node(step_type="run") as n:
        n.add_meta("run_id", i)
        n.add_meta("accuracy", round(float(rng.random()), 4))
        if i < N_ART:
            df = make_csv(ROWS)
            df.to_csv(n / "data.csv", index=False)  # stored (chunked)
            p = BASE / f"run_{i}.csv"
            df.to_csv(p, index=False)  # plain copy
            target_ids.append(n.node_id)
            direct_paths.append(p)

art_mb = direct_paths[0].stat().st_size / 1e6
print(
    f"built {N_TOTAL} nodes ({N_ART} with ~{art_mb:.1f} MB CSVs) in "
    f"{time.perf_counter() - t0:.0f}s"
)
# A node is a database row now, not a folder: the artifact bytes live in the
# deduplicated chunk pool inside ancestree.db. The whole store root is just:
print("store root holds only:", sorted(p.name for p in ROOT.iterdir()))
print("store stats:", store.stats())

built 1000 nodes (30 with ~2.2 MB CSVs) in 49s
store root holds only: ['.scratch', 'ancestree.db', 'ancestree.db-shm', 'ancestree.db-wal']
store stats: {'nodes': 1000, 'artifacts': 30, 'chunks': 1636, 'logical_bytes': 64585656, 'chunk_plain_bytes': 64585656, 'chunk_stored_bytes': 23994110, 'database_bytes': 27660288, 'dedup_ratio': 2.692}


## 2. Baseline &mdash; the file is already at a known path

No store, no lineage, no reassembly: just read the bytes / parse the CSV sitting on disk. This is the bar everything else is compared against.

In [3]:
direct_bytes = timed("direct  path.read_bytes()", lambda: direct_paths[0].read_bytes())
direct_parse = timed("direct  pd.read_csv(path)", lambda: pd.read_csv(direct_paths[0]))

direct  path.read_bytes()                    median    0.097 ms   (best 0.088)
direct  pd.read_csv(path)                    median   15.872 ms   (best 15.689)


## 3. Retrieve through the store, by known id &mdash; every step

You already know the node id. Time each step: `get` (loads the node row), `.artifacts()` (discovers the file and, on a cold cache, reassembles it from chunks), then `pd.read_csv` on the returned path. Cold clears the session read cache first so every artifact is a fresh miss; warm runs straight after, all cache hits. (`get` was `get_node` in 0.1.x; there is no public cache-clear verb, so we reach into the store's chunk cache directly to force a cold read.)

In [4]:
def breakdown_by_id(clear):
    if clear:
        store._chunks.clear_cache()  # drop the session read cache -> cold reassembly
    g = a = r = 0.0
    for tid in target_ids:
        t = time.perf_counter()
        node = store.get(tid)
        g += time.perf_counter() - t
        t = time.perf_counter()
        path = node.artifacts("data.csv")[0]
        a += time.perf_counter() - t
        t = time.perf_counter()
        _ = pd.read_csv(path)
        r += time.perf_counter() - t
    n = len(target_ids)
    return g / n * 1000, a / n * 1000, r / n * 1000


id_cold = breakdown_by_id(clear=True)  # .artifacts() reassembles from the pool
id_warm = breakdown_by_id(clear=False)  # .artifacts() is a cache hit
for label, (g, a, r) in [("COLD (cache miss)", id_cold), ("WARM (cache hit)", id_warm)]:
    print(
        f"{label:<20}  get {g:7.3f}  | artifacts() {a:7.3f}  | read_csv {r:7.3f}  "
        f"| TOTAL {g + a + r:7.3f} ms"
    )

COLD (cache miss)     get   0.043  | artifacts()   7.023  | read_csv  16.255  | TOTAL  23.321 ms
WARM (cache hit)      get   0.024  | artifacts()   0.097  | read_csv  16.035  | TOTAL  16.156 ms


## 4. Retrieve through the store, by metadata search &mdash; every step

The realistic case where you *don't* know the id: `find(run_id=...)` scans the index, then the same discover + read. Shows what the search itself adds on top of section 3. (`find` was `find_node` in 0.1.x.)

In [5]:
def breakdown_by_search(clear):
    if clear:
        store._chunks.clear_cache()
    s = a = r = 0.0
    for i, _ in enumerate(target_ids):
        t = time.perf_counter()
        node = store.find(run_id=i)[0]
        s += time.perf_counter() - t
        t = time.perf_counter()
        path = node.artifacts("data.csv")[0]
        a += time.perf_counter() - t
        t = time.perf_counter()
        _ = pd.read_csv(path)
        r += time.perf_counter() - t
    n = len(target_ids)
    return s / n * 1000, a / n * 1000, r / n * 1000


se_cold = breakdown_by_search(clear=True)
se_warm = breakdown_by_search(clear=False)
for label, (s, a, r) in [("COLD", se_cold), ("WARM", se_warm)]:
    print(
        f"{label:<6} find {s:7.3f}  | artifacts() {a:7.3f}  | read_csv {r:7.3f}  "
        f"| TOTAL {s + a + r:7.3f} ms"
    )

COLD   find   0.064  | artifacts()   7.026  | read_csv  16.299  | TOTAL  23.389 ms
WARM   find   0.041  | artifacts()   0.096  | read_csv  16.005  | TOTAL  16.142 ms


## 5. The convenience path: `node / "data.csv"`

When you know the filename you skip `.artifacts()` and index it directly. Same materialization underneath.

In [6]:
def by_slash(clear):
    if clear:
        store._chunks.clear_cache()
    tot = 0.0
    for tid in target_ids:
        t = time.perf_counter()
        _ = pd.read_csv(store.get(tid) / "data.csv")
        tot += time.perf_counter() - t
    return tot / len(target_ids) * 1000


print(
    f"node / 'data.csv' then read  COLD {by_slash(True):7.3f} ms   "
    f"WARM {by_slash(False):7.3f} ms"
)

node / 'data.csv' then read  COLD  23.173 ms   WARM  16.297 ms


## 6. Summary &mdash; store overhead over a direct path read

In [7]:
def row(name, total):
    print(f"  {name:<34} {total:8.3f} ms   ({total / direct_parse:5.1f}x direct)")


print(f"Reading one ~{art_mb:.1f} MB CSV ({len(target_ids)} samples averaged)\n")
print(f"  {'direct pd.read_csv(path)':<34} {direct_parse:8.3f} ms   (baseline)")
row("store by id   — COLD (first touch)", sum(id_cold))
row("store by id   — WARM (cache hit)", sum(id_warm))
row("store by search — COLD", sum(se_cold))
row("store by search — WARM", sum(se_warm))
print()
print(
    f"  decompression cost (artifacts COLD - WARM): {id_cold[1] - id_warm[1]:.3f} ms / artifact"
)
print(
    f"  index search cost  (find):                  {se_cold[0]:.3f} ms over {N_TOTAL} nodes"
)
print(f"  node-row load      (get):                   {id_cold[0]:.3f} ms")
print()
print("Takeaways:")
print(
    " - WARM store reads ~= a direct read: once cached, the store adds almost nothing."
)
print(
    " - The COLD premium is the chunk reassembly, paid once per artifact per session."
)
print(
    " - Finding a node (id load or index search) is sub-millisecond and dwarfed by I/O."
)

Reading one ~2.2 MB CSV (30 samples averaged)

  direct pd.read_csv(path)             15.872 ms   (baseline)
  store by id   — COLD (first touch)   23.321 ms   (  1.5x direct)
  store by id   — WARM (cache hit)     16.156 ms   (  1.0x direct)
  store by search — COLD               23.389 ms   (  1.5x direct)
  store by search — WARM               16.142 ms   (  1.0x direct)

  decompression cost (artifacts COLD - WARM): 6.926 ms / artifact
  index search cost  (find):                  0.064 ms over 1000 nodes
  node-row load      (get):                   0.043 ms

Takeaways:
 - WARM store reads ~= a direct read: once cached, the store adds almost nothing.
 - The COLD premium is the chunk reassembly, paid once per artifact per session.
 - Finding a node (id load or index search) is sub-millisecond and dwarfed by I/O.


## 7. Read/write penalty by data type and size

The headline table: for a spread of file types and sizes, how long does it take to **write** and **read** the file *directly* (a plain path) versus *through the store*?

- **W store** = the full `create_node` block: the native write **plus** the synchronous packing that happens at block exit &mdash; the artifact is chunked (Layer-1 CDC) and folded into the pool before the block returns (AD4). Packing is *not* deferred to a background worker; the write you issue is the write you wait for. Provenance capture is a fixed per-node cost (near-zero here, off a repo).
- **R cold** = first touch in a session (reassemble every chunk into the read cache).
- **R warm** = a second read in the same session (cache hit).

All `x` columns are the multiple over the *direct* operation. The largest rows take a few seconds (packing thousands of chunks).

In [8]:
import pickle

DT_ROOT = WORKDIR / "dt_bench_store"
DT_BASE = WORKDIR / "dt_bench_direct"
DT_BASE.mkdir(parents=True)
dt_store = ancestree.LineageStore(DT_ROOT, dedup=False, chunk=True)
dt_rng = np.random.default_rng(0)


def _med(fn, reps):
    ts = []
    for _ in range(reps):
        t = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t) * 1000)
    return float(np.median(ts))


def _npy_w(p, d):
    with open(p, "wb") as f:
        np.save(f, d)


def _npy_r(p):
    with open(p, "rb") as f:
        return np.load(f)


def _npz_w(p, d):
    with open(p, "wb") as f:
        np.savez_compressed(f, a=d)


def _npz_r(p):
    with open(p, "rb") as f:
        return np.load(f)["a"]


FORMATS = {
    "bytes": {
        "fn": "d.bin",
        "make": lambda a: a.tobytes(),
        "w": lambda p, d: Path(p).write_bytes(d),
        "r": lambda p: Path(p).read_bytes(),
    },
    "npy": {"fn": "d.npy", "make": lambda a: a, "w": _npy_w, "r": _npy_r},
    "npz": {"fn": "d.npz", "make": lambda a: a, "w": _npz_w, "r": _npz_r},
    "csv": {
        "fn": "d.csv",
        "make": lambda a: pd.DataFrame(
            a[: len(a) // 4 * 4].reshape(-1, 4), columns=list("abcd")
        ),
        "w": lambda p, d: d.to_csv(p, index=False),
        "r": lambda p: pd.read_csv(p),
    },
    "pickle": {
        "fn": "d.pkl",
        "make": lambda a: a,
        "w": lambda p, d: Path(p).write_bytes(pickle.dumps(d)),
        "r": lambda p: pickle.loads(Path(p).read_bytes()),
    },
}
SIZES_MB = [
    0.25,
    1,
    4,
    16,
    64,
]  # underlying float64 array size; file sizes differ per format

rows = []
for size_mb in SIZES_MB:
    arr = dt_rng.standard_normal(int(size_mb * 1e6 / 8))
    for name, F in FORMATS.items():
        data = F["make"](arr)
        dpath = DT_BASE / f"{name}_{size_mb}_{F['fn']}"
        wd = _med(lambda: F["w"](dpath, data), 5)  # direct write
        file_mb = dpath.stat().st_size / 1e6
        rd = _med(lambda: F["r"](dpath), 7)  # direct read
        # store write = the full create_node block: native write + synchronous
        # chunking into the pool at block exit (a single sample, like the write).
        t0 = time.perf_counter()
        with dt_store.create_node(step_type="blob") as node:
            F["w"](node / F["fn"], data)
        ws = (time.perf_counter() - t0) * 1000
        nid = node.node_id

        def _cold():
            dt_store._chunks.clear_cache()
            return F["r"](dt_store.get(nid) / F["fn"])

        rc = _med(_cold, 5)  # cold read (materialize + parse)
        rw = _med(
            lambda: F["r"](dt_store.get(nid) / F["fn"]), 7
        )  # warm read (cache hit)
        rows.append({"type": name, "file_mb": file_mb, "wd": wd, "ws": ws, "rd": rd, "rc": rc, "rw": rw})
    print(f"  done size {size_mb} MB")
print("measurements complete")

  done size 0.25 MB
  done size 1 MB
  done size 4 MB
  done size 16 MB
  done size 64 MB
measurements complete


### The table

In [9]:
hdr = (
    f"{'type':7} {'file MB':>8} | {'W direct':>9} {'W store':>9} {'W x':>6} | "
    f"{'R direct':>9} {'R cold':>8} {'R warm':>8} {'cold x':>7} {'warm x':>7}"
)
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(
        f"{r['type']:7} {r['file_mb']:8.2f} | {r['wd']:9.2f} {r['ws']:9.2f} "
        f"{r['ws'] / r['wd']:5.0f}x | {r['rd']:9.2f} {r['rc']:8.2f} {r['rw']:8.2f} "
        f"{r['rc'] / r['rd']:6.0f}x {r['rw'] / r['rd']:6.1f}x"
    )

print(
    "\nReadings are milliseconds (median; W store a single sample). Packing is "
    "synchronous at block exit."
)

type     file MB |  W direct   W store    W x |  R direct   R cold   R warm  cold x  warm x
-------------------------------------------------------------------------------------------
bytes       0.25 |      0.09     64.61   681x |      0.03     1.66     0.06     49x    1.9x
npy         0.25 |      0.13     56.97   444x |      0.06     1.88     0.11     29x    1.7x
npz         0.24 |      7.49     66.59     9x |      0.77     1.80     0.80      2x    1.0x
csv         0.61 |     23.09    148.28     6x |      2.98     5.80     3.09      2x    1.0x
pickle      0.25 |      0.14     57.05   417x |      0.04     1.81     0.08     41x    1.9x
bytes       1.00 |      0.26    170.07   649x |      0.06     4.80     0.09     87x    1.6x
npy         1.00 |      0.25    137.37   544x |      0.08     4.86     0.12     59x    1.4x
npz         0.96 |     31.19    191.42     6x |      2.77     4.97     2.81      2x    1.0x
csv         2.45 |     92.03    515.01     6x |     11.02    20.16    11.00     

### Takeaways

In [10]:
fast = [r for r in rows if r["type"] in ("bytes", "npy", "pickle")]
slow = [r for r in rows if r["type"] == "csv"]
print("WRITE penalty is the big one:")
print(
    f"  packing turns a ~{min(r['wd'] for r in fast):.2f}-{max(r['wd'] for r in fast):.1f} ms direct write into"
)
print(
    f"  {min(r['ws'] for r in fast):.0f}-{max(r['ws'] for r in fast):.0f} ms — up to {max(r['ws'] / r['wd'] for r in fast):.0f}x — because each ~32 KB chunk"
)
print(
    "  is separately hashed, compressed and inserted into the pool. Bigger artifacts pay proportionally more."
)
print()
print("READ penalty depends entirely on the format:")
print(
    f"  fast binary (npy/bytes/pickle): cold read up to {max(r['rc'] / r['rd'] for r in fast):.0f}x direct"
)
print("     (the reassembly dwarfs the ~0-1 ms raw read), warm ~1x once cached.")
print(
    f"  slow text (csv): cold only ~{max(r['rc'] / r['rd'] for r in slow):.1f}x — the parse hides the reassembly."
)
print()
print("So: the store is cheap when your I/O is already slow (csv), and expensive")
print("relative to fast formats (npy) — especially on WRITE. Warm re-reads are ~free.")

WRITE penalty is the big one:
  packing turns a ~0.09-20.0 ms direct write into
  57-9456 ms — up to 932x — because each ~32 KB chunk
  is separately hashed, compressed and inserted into the pool. Bigger artifacts pay proportionally more.

READ penalty depends entirely on the format:
  fast binary (npy/bytes/pickle): cold read up to 113x direct
     (the reassembly dwarfs the ~0-1 ms raw read), warm ~1x once cached.
  slow text (csv): cold only ~1.9x — the parse hides the reassembly.

So: the store is cheap when your I/O is already slow (csv), and expensive
relative to fast formats (npy) — especially on WRITE. Warm re-reads are ~free.


## 8. Clean up

Close the stores (releasing their connections and read caches) and delete the temp workdir.

In [11]:
store.close()
dt_store.close()
shutil.rmtree(WORKDIR, ignore_errors=True)